# Big Data Final Project: NYC Yellow Taxi Analysis
## Application: MongoDB | Architecture: Medallion (Bronze, Silver, Gold)

In [ ]:
!pip install pandas pymongo dnspython matplotlib

## This cell imports the required modules ,establishes a connection to the local MangoDB instance and initializes Taxiproject database and 1st_stage Collection.

In [ ]:
import pandas as pd
from pymongo import MongoClient
import matplotlib.pyplot as plt

client = MongoClient("mongodb://localhost:27017/")
db = client["TaxiProject"]
bronze_col = db["1st_stage"]



##Bronze Layer:Raw Data Ingestion

##This cell reads the raw NYC TAXI csv file converts the first 55000 rows into a dictionary format and ingests them into Bronze collection in MangoDB

In [ ]:
df_raw = pd.read_csv("yellow_tripdata_2015-01.csv",nrows = 55000)
df_raw = df_raw.head(55000)
data_dict = df_raw.to_dict("records")
bronze_col.insert_many(data_dict)

print(f"Bronze Layer: {bronze_col.count_documents({})} rows ingested.")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

##Silver Layer: Data Cleaning and Transformation

##This cell retrieves data from the Bronze layer to perform cleaning tasks: removing duplicates, dropping rows with missing essential values, and converting date strings into datetime objects. The cleaned data is then stored in the 2nd_stage (Silver) collection.

In [ ]:
silver_col = db["2nd_stage"]
raw_data = list(bronze_col.find())
df_silver =pd.DataFrame(raw_data)

df_silver.drop_duplicates(inplace=True)

df_silver.dropna(subset=['fare_amount','passenger_count','trip_distance'],inplace = True)
df_silver['tpep_pickup_datetime'] = pd.to_datetime(df_silver['tpep_pickup_datetime'])
df_silver['tpep_dropoff_datetime'] = pd.to_datetime(df_silver['tpep_dropoff_datetime'])

silver_col.delete_many({})
silver_col.insert_many(df_silver.to_dict("records"))

print(f"Second stage: {silver_col.count_documents({})} cleaned rows stored.")

df_silver = df_raw.copy()

In [ ]:
print("Before cleaning:", len(df_raw))
print("After cleaning:", len(df_silver))

##Gold Layer: Aggregation and Business Logic
##This cell uses a MongoDB aggregation pipeline to calculate business metrics: total trips and average fare amount per VendorID. The results represent the "Gold" or presentation layer of the data.

In [ ]:

pipeline = [
    {
        "$group": {
            "_id": "$VendorID",
            "total_trips": {"$sum": 1},
            "avg_fare": {"$avg": "$fare_amount"}
        }
    },
    {"$sort": {"total_trips": -1}}
]

gold_results = list(silver_col.aggregate(pipeline))

# Show the presentation layer results
for result in gold_results:
    print(result)

In [ ]:
vendors = [str(r['_id']) for r in gold_results]
trips = [r['total_trips'] for r in gold_results]

plt.figure(figsize=(6,4))
plt.bar(vendors,trips,color='blue', edgecolor ='black')
plt.xlabel('Vendor ID')
plt.ylabel('Number of Trips')
plt.title('Total Trips per vendor (Gold Layer Insights)')
plt.show()

#The below line chart show how fare amounts fluctuate throughout the day,providing a clear temporal trend

In [ ]:
# Aggregate average fare by pickup hour
pipeline_line = [
    {
        "$group": {
            "_id": {"$hour": "$tpep_pickup_datetime"},
            "avg_fare": {"$avg": "$fare_amount"}
        }
    },
    {"$sort": {"_id": 1}}
]

line_results = list(silver_col.aggregate(pipeline_line))
hours = [r['_id'] for r in line_results]
avg_fares = [r['avg_fare'] for r in line_results]

plt.figure(figsize=(10,6))
plt.plot(hours, avg_fares, marker='o', linestyle='-', color='green')
plt.xlabel('Hour of Day (24h format)')
plt.ylabel('Average Fare ($)')
plt.title('Trend: Average Fare Amount by Hour')
plt.grid(True)
plt.show()

##This final cell prepares the Silver layer data for spatial analysis by extracting pickup coordinates and hours. It then uses folium to create an interactive animated heatmap showing taxi pickup density across New York City over a 24-hour cycle.

In [ ]:
import folium
from folium.plugins import HeatMapWithTime

df_silver['tpep_pickup_datetime'] = pd.to_datetime(
    df_silver['tpep_pickup_datetime'], errors='coerce'
)

df_silver['hour'] = df_silver['tpep_pickup_datetime'].dt.hour


heat_data = []
for hour in range(24):
    subset = df_silver[df_silver['hour'] == hour]
    heat_data.append(subset[['pickup_latitude', 'pickup_longitude']].values.tolist())

# Creation for map:
m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB dark_matter')

HeatMapWithTime(
    heat_data,
    radius=8,
    auto_play=True,
    max_opacity=0.8
).add_to(m)

m